### Import

In [1]:
import torch
import torchvision
print("Torch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

with open("prompt.txt", "r", encoding="utf-8") as f:
    instruction= f.read()


# Cuz Titron is a chud
import os
cl_dir = r"C:\Program Files (x86)\Microsoft Visual Studio\18\BuildTools\VC\Tools\MSVC\14.51.36231\bin\Hostx64\x64"
os.environ["PATH"] = cl_dir + ";" + os.environ["PATH"]
os.environ["CC"] = cl_dir + r"\cl.exe"
os.environ["CXX"] = cl_dir + r"\cl.exe"

Torch: 2.8.0+cu129
TorchVision: 0.23.0+cu129
Using device: cuda


In [2]:
!nvidia-smi


Tue Oct 20 09:20:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 556.19                 Driver Version: 556.19         CUDA Version: 12.5     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...  WDDM  |   00000000:01:00.0  On |                  N/A |
| N/A   54C    P8              1W /   80W |     226MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import torch

print("Torch:", torch.__version__)
print("CUDA compiled into PyTorch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Torch: 2.8.0+cu129
CUDA compiled into PyTorch: 12.9
CUDA available: True
CUDA device count: 1
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [4]:
import sys
print(sys.executable)
!{sys.executable} -m pip show torch



C:\Users\joel.sutanto\100426\my_unsloth\Scripts\python.exe
Name: torch
Version: 2.8.0+cu129
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3-Clause
Location: C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages
Requires: filelock, fsspec, jinja2, networkx, setuptools, sympy, typing-extensions
Required-by: accelerate, bitsandbytes, cut-cross-entropy, peft, torchvision, unsloth, unsloth_zoo, xformers


In [5]:
import torch

print("Torch:", torch.__version__)
print("CUDA compiled:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    

Torch: 2.8.0+cu129
CUDA compiled: 12.9
CUDA available: True
GPU count: 1
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
import unsloth
print(unsloth.__version__)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1020 09:21:58.249000 29316 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
[bitsandbytes.cextension|WARNING]No prebuilt binary for CUDA 12.9, loading CUDA 12.8 instead. Set BNB_CUDA_VERSION to override.


🦥 Unsloth Zoo will now patch everything to make training faster!
2026.9.14


In [7]:
import datasets
import dill
import pyarrow

print("datasets:", datasets.__version__)
print("dill:", dill.__version__)
print("pyarrow:", pyarrow.__version__)

datasets: 4.3.0
dill: 0.4.0
pyarrow: 25.0.1


In [8]:
import json
from pathlib import Path
from unsloth import FastVisionModel

# model, processor = FastVisionModel.from_pretrained(
#     model_name="unsloth/Qwen2-VL-2B-Instruct-bnb-4bit",
#     load_in_4bit=True,
#     use_gradient_checkpointing="unsloth",
# )

# model = FastVisionModel.for_training(model)
# print(f"Model ready on {device}")

model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit=True,
    max_seq_length=1024,   # added
    use_gradient_checkpointing="unsloth",
)

==((====))==  Unsloth 2026.9.14: Fast Qwen3_5 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 4060 Laptop GPU. Num GPUs = 1. Max memory: 7.996 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.8.0+cu129. CUDA: 8.9. CUDA Toolkit: 12.9. Triton: 3.8.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 473/473 [00:03<00:00, 156.38it/s]


### Train-test-split

In [9]:

from datasets import Dataset
from PIL import Image
import json
import os
import re

# =========================
# 1. Load ground truth
# =========================

with open("dataset/labels.json", "r", encoding="utf-8") as f:
    labels = json.load(f)


# =========================
# 2. Function to identify
#    the original image
# =========================

def get_base_name(filename):
    name = os.path.splitext(filename)[0].lower()

    # image_1 -> image1
    name = re.sub(r"image_(\d+)", r"image\1", name)

    # image1_blur -> image1
    name = re.sub(r"_(blur|glare|lowq)$", "", name)

    return name


# =========================
# 3. Create ground-truth
#    lookup table
# =========================

ground_truth = {}

for filename, value in labels.items():
    base_name = get_base_name(filename)

    # If your JSON has {"fields": {...}}
    if isinstance(value, dict) and "fields" in value:
        ground_truth[base_name] = value["fields"]

    # If your JSON directly contains the fields
    else:
        ground_truth[base_name] = value


# =========================
# 4. Find all images
# =========================

image_files = [
    f for f in os.listdir("dataset")
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]


# =========================
# 5. Match images to labels
# =========================

images = []
texts = []
bases = []   # original-image ID (blur/glare/lowq variants share one)

for filename in sorted(image_files):

    base_name = get_base_name(filename)

    if base_name not in ground_truth:
        print(f"⚠️ No ground truth found for: {filename}")
        continue

    image_path = os.path.join("dataset", filename)

    images.append(Image.open(image_path).convert("RGB"))

    # Convert ground truth dictionary into text
    texts.append(json.dumps(ground_truth[base_name]))
    bases.append(base_name)


# =========================
# 6. Create HuggingFace Dataset
# =========================

data = {
    "image": images,
    "text": texts,
    "base": bases,
}

dataset = Dataset.from_dict(data)

print(dataset)

⚠️ No ground truth found for: image65.jpg
⚠️ No ground truth found for: image65_blur.jpg
⚠️ No ground truth found for: image65_glare.jpg
⚠️ No ground truth found for: image65_lowq.jpg
Dataset({
    features: ['image', 'text', 'base'],
    num_rows: 380
})


In [10]:
import datasets
import dill
import pyarrow

print("datasets:", datasets.__version__)
print("dill:", dill.__version__)
print("pyarrow:", pyarrow.__version__)

datasets: 4.3.0
dill: 0.4.0
pyarrow: 25.0.1


In [11]:

# Split it - 80/20 is standard, adjust if needed
# Group split: every variant of an original image (image1, image1_blur, image1_glare, ...)
# goes to the SAME side, so the test set never contains a re-degraded copy of a training image.
import random
bases_unique = sorted(set(dataset["base"]))
random.Random(42).shuffle(bases_unique)
test_bases = set(bases_unique[: int(len(bases_unique) * 0.2)])

all_bases = dataset["base"]
train_idx = [i for i, b in enumerate(all_bases) if b not in test_bases]
test_idx  = [i for i, b in enumerate(all_bases) if b in test_bases]

train_dataset = dataset.select(train_idx)
test_dataset  = dataset.select(test_idx)
print(f"{len(train_dataset)} train | {len(test_dataset)} test | {len(bases_unique)} original images")

# For QLoRA fine-tuning with Qwen-VL, you'll need a collate function
def collate_fn(batch):
    images = [item['image'] for item in batch]
    texts = [item['text'] for item in batch]
    # Process based on your model's processor
    return {'images': images, 'texts': texts}

# Then pass to DataLoader
from torch.utils.data import DataLoader
train_loader = DataLoader(train_dataset, batch_size=4, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=4, collate_fn=collate_fn)

304 train | 76 test | 95 original images


In [12]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers     = True, # False if not finetuning vision layers
    finetune_language_layers   = True, # False if not finetuning language layers
    finetune_attention_modules = True, # False if not finetuning attention layers
    finetune_mlp_modules       = True, # False if not finetuning MLP layers

    r = 16,           # The larger, the higher the accuracy, but might overfit
    lora_alpha = 16,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
    # target_modules = "all-linear", # Optional now! Can specify a list if needed
)

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


In [13]:
def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : instruction},
            {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample["text"]} ]
        },
    ]
    return { "messages" : conversation }
pass

In [14]:
converted_dataset = [convert_to_conversation(sample) for sample in dataset]

In [15]:
FastVisionModel.for_inference(model) # Enable for inference!

image = dataset[2]["image"]

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 1.5, min_p = 0.1)

C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages\unsloth\import_fixes.py:5429: UserWarning: 'has_cuda' is deprecated, please use 'torch.backends.cuda.is_built()'
  return original(name)
C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages\unsloth\import_fixes.py:5429: UserWarning: 'has_cudnn' is deprecated, please use 'torch.backends.cudnn.is_available()'
  return original(name)
C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages\unsloth\import_fixes.py:5429: UserWarning: 'has_mps' is deprecated, please use 'torch.backends.mps.is_built()'
  return original(name)
C:\Users\joel.sutanto\100426\my_unsloth\Lib\site-packages\unsloth\import_fixes.py:5429: UserWarning: 'has_mkldnn' is deprecated, please use 'torch.backends.mkldnn.is_available()'
  return original(name)
[unsloth_zoo.log|WARNING]Unsloth: torch.compile could not generate code for Qwen3_5VisionPatchEmbed_forward; running it eagerly from here. Training is unaffected apart from speed. (BackendCompilerFailed)
[uns

```json
{
    "Id-no": "01",
    "Name": "Warganegara Lelaki",
    "Address1": "Jalan Lanjut 9",
    "Address2": "Taman Desa Cemerlang",
    "Address3": "Johor",
    "Postal_Code": "81800",
    "State": "Johor",
    "Gender": "Lelaki"
}
```<|im_end|>
<|endoftext|>


In [16]:
from transformers import TextStreamer

text_streamer = TextStreamer(tokenizer, skip_prompt=True)

_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.5,
    min_p=0.1,
)

```json
{
    "Id-no": "01",
    "Name": "Warganegara Lelaki",
    "Address1": "Jalan Lanjut 9",
    "Address2": "Taman Desa Cemerlang",
    "Address3": "Johor",
    "Postal_Code": "81800",
    "State": "Johor",
    "Gender": "Lelaki"
}
```<|im_end|>
<|endoftext|>


### Training 

In [17]:
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

### Epoch sweep (5 / 10 / 20 / 50)
Each epoch count trains in its **own Python process**, so the GPU is fully released between runs (deleting the model inside a notebook never frees everything, which caused the "No GPU memory" error). Then plot, pick a winner, set `BEST_EPOCHS` in the final training cell.

In [18]:
%%writefile sweep_train.py
import unsloth   # must come before transformers / trl / peft
import gc, json, sys
import torch
from unsloth import FastVisionModel
from transformers import TrainerCallback
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

MODEL_NAME = "unsloth/Qwen3.5-0.8B"


class MemCB(TrainerCallback):
    """Prints VRAM per step and empties the CUDA cache each step.
    The empty_cache() is what keeps Unsloth's fused cross-entropy from running out of 'free' memory."""
    def on_step_end(self, args, state, control, **kw):
        alloc = torch.cuda.memory_allocated() / 1e9
        peak = torch.cuda.max_memory_allocated() / 1e9
        gc.collect(); torch.cuda.empty_cache()
        free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"step {state.global_step}/{state.max_steps} | alloc {alloc:.2f} GB | peak {peak:.2f} GB | free {free:.2f} GB", flush=True)


def convert_to_conversation(sample, instruction):
    return {"messages": [
        {"role": "user", "content": [
            {"type": "text", "text": instruction},
            {"type": "image", "image": sample["image"]}]},
        {"role": "assistant", "content": [
            {"type": "text", "text": sample["text"]}]},
    ]}


def build_model():
    """Fresh base model + fresh LoRA adapter (same settings as the notebook)."""
    model, tokenizer = FastVisionModel.from_pretrained(
        MODEL_NAME,
        load_in_4bit=True,
        max_seq_length=1024,
        use_gradient_checkpointing="unsloth",
    )
    model = FastVisionModel.get_peft_model(
        model,
        finetune_vision_layers=True,
        finetune_language_layers=True,
        finetune_attention_modules=True,
        finetune_mlp_modules=True,
        r=16,
        lora_alpha=16,
        lora_dropout=0,
        bias="none",
        random_state=3407,
        use_rslora=False,
        loftq_config=None,
    )
    FastVisionModel.for_training(model)
    return model, tokenizer


def make_trainer(model, tokenizer, train_conv, eval_conv, n_epochs, out_dir, quiet=False, mem_log=False):
    return SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        data_collator=UnslothVisionDataCollator(model, tokenizer, max_seq_length=1024),
        train_dataset=train_conv,
        eval_dataset=eval_conv,
        callbacks=[MemCB()] if mem_log else None,
        args=SFTConfig(
            per_device_train_batch_size=1,
            per_device_eval_batch_size=1,
            gradient_accumulation_steps=8,
            warmup_steps=5,
            num_train_epochs=n_epochs,
            learning_rate=2e-4,
            logging_steps=1,
            optim="adamw_8bit",
            weight_decay=0.001,
            lr_scheduler_type="linear",
            seed=3407,
            output_dir=out_dir,
            report_to="none",
            disable_tqdm=quiet,

            eval_strategy="epoch",      # eval loss at the end of every epoch
            save_strategy="no",         # no checkpoints -> no disk spam

            remove_unused_columns=False,
            dataset_text_field="",
            dataset_kwargs={"skip_prepare_dataset": True},
            max_length=1024,
        ),
    )


def main():
    from datasets import load_from_disk
    n_epochs = int(sys.argv[1])

    train_ds = load_from_disk("sweep_data/train")
    eval_ds = load_from_disk("sweep_data/test")
    with open("sweep_data/instruction.txt", encoding="utf-8") as f:
        instruction = f.read()

    train_conv = [convert_to_conversation(s, instruction) for s in train_ds]
    eval_conv = [convert_to_conversation(s, instruction) for s in eval_ds]
    print(f"{len(train_conv)} train | {len(eval_conv)} eval | {n_epochs} epochs", flush=True)

    model, tokenizer = build_model()
    trainer = make_trainer(model, tokenizer, train_conv, eval_conv, n_epochs,
                           f"outputs/sweep_{n_epochs}", quiet=True, mem_log=True)
    trainer.train()

    with open(f"sweep_log_{n_epochs}_epochs.json", "w") as f:
        json.dump(trainer.state.log_history, f)
    print("finished", n_epochs, "epochs", flush=True)


if __name__ == "__main__":
    main()


Overwriting sweep_train.py


In [19]:
import gc, json, os, shutil, sys, time, subprocess
import torch

# Conversation format for the final training cell (uses convert_to_conversation from cell 14)
train_conv = [convert_to_conversation(s) for s in train_dataset]
eval_conv  = [convert_to_conversation(s) for s in test_dataset]
print(len(train_conv), "train |", len(eval_conv), "eval")

# Hand the exact same split to the sweep subprocesses
shutil.rmtree("sweep_data", ignore_errors=True)
train_dataset.save_to_disk("sweep_data/train")
test_dataset.save_to_disk("sweep_data/test")
with open("sweep_data/instruction.txt", "w", encoding="utf-8") as f:
    f.write(instruction)
print("saved split to sweep_data/")


304 train | 76 eval


Saving the dataset (1/1 shards): 100%|██████████| 76/76 [00:00<00:00, 181.88 examples/s]

saved split to sweep_data/


In [ ]:
EPOCH_OPTIONS = [5, 10, 20, 50]

# Release everything this kernel still holds on the GPU (model, trainer, old tensors, old tracebacks)
for name in ("trainer", "model", "tokenizer", "inputs", "outputs", "batch",
             "trainer_stats", "text_streamer", "_"):
    globals().pop(name, None)
sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()
torch.cuda.empty_cache()
free, total = torch.cuda.mem_get_info()
print(f"Free VRAM before sweep: {free/1024**3:.2f} / {total/1024**3:.2f} GB")

env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")


def echo(line):
    """Show only the useful lines in the notebook (everything is in the .log file)."""
    if "eval_loss" in line or " train | " in line or "finished" in line:
        print(line.strip())
    elif line.startswith("step "):      # VRAM line for every step
        print(line.strip())


for n in EPOCH_OPTIONS:
    log_json = f"sweep_log_{n}_epochs.json"
    if os.path.exists(log_json):    # resume: finished runs are skipped (delete the json to rerun)
        print(f"[skip] {n} epochs - {log_json} exists")
        continue

    print(f"\n===== {n} epochs =====")
    log_txt = f"sweep_run_{n}_epochs.log"
    t0 = time.time()

    # The child writes straight to the file; we tail the file. (Reading a pipe can hang forever if a
    # compiler helper process keeps the pipe open after training has already crashed.)
    with open(log_txt, "wb") as lf:
        proc = subprocess.Popen([sys.executable, "sweep_train.py", str(n)],
                                stdout=lf, stderr=subprocess.STDOUT, env=env)

    with open(log_txt, "r", encoding="utf-8", errors="replace") as rf:
        while True:
            pos = rf.tell()
            line = rf.readline()
            if line and line.endswith("\n"):
                echo(line)
            elif proc.poll() is not None:
                for rest in rf:                  # drain whatever is left
                    echo(rest)
                break
            else:
                rf.seek(pos)                     # partial line: wait for the rest
                time.sleep(1)

    if proc.returncode != 0:
        print("\n".join(open(log_txt, encoding="utf-8", errors="replace").read().splitlines()[-25:]))
        raise RuntimeError(f"{n}-epoch run failed - full output in {log_txt}")
    print(f"done in {(time.time() - t0) / 60:.1f} min")

results = {n: json.load(open(f"sweep_log_{n}_epochs.json"))
           for n in EPOCH_OPTIONS if os.path.exists(f"sweep_log_{n}_epochs.json")}
print("\nSweep finished:", list(results))


Free VRAM before sweep: 6.11 / 8.00 GB
[skip] 5 epochs - sweep_log_5_epochs.json exists
[skip] 10 epochs - sweep_log_10_epochs.json exists

===== 20 epochs =====
304 train | 76 eval | 20 epochs
step 1/760 | alloc 0.99 GB | peak 3.54 GB | free 6.31 GB
step 2/760 | alloc 0.99 GB | peak 3.54 GB | free 6.30 GB
step 3/760 | alloc 0.99 GB | peak 3.54 GB | free 6.30 GB
step 4/760 | alloc 1.00 GB | peak 3.54 GB | free 6.26 GB
step 5/760 | alloc 1.00 GB | peak 3.54 GB | free 6.28 GB
step 6/760 | alloc 1.00 GB | peak 3.54 GB | free 6.28 GB
step 7/760 | alloc 1.00 GB | peak 3.54 GB | free 6.28 GB
step 8/760 | alloc 1.00 GB | peak 3.57 GB | free 6.28 GB
step 9/760 | alloc 0.99 GB | peak 3.57 GB | free 6.28 GB
step 10/760 | alloc 1.01 GB | peak 3.57 GB | free 6.28 GB
step 11/760 | alloc 1.01 GB | peak 3.57 GB | free 6.26 GB
step 12/760 | alloc 1.01 GB | peak 3.57 GB | free 6.24 GB
step 13/760 | alloc 1.00 GB | peak 3.57 GB | free 6.26 GB
step 14/760 | alloc 0.99 GB | peak 3.57 GB | free 6.26 GB
ste

In [ ]:
import matplotlib.pyplot as plt
from collections import defaultdict

# Summary table
for n_epochs, log in results.items():
    ev = [(l["epoch"], l["eval_loss"]) for l in log if "eval_loss" in l]
    best_ep, best_loss = min(ev, key=lambda t: t[1])
    print(f"{n_epochs:>2}-epoch run | best eval loss {best_loss:.4f} @ epoch {best_ep:.0f} | final eval loss {ev[-1][1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for n_epochs, log in results.items():
    ev = [(l["epoch"], l["eval_loss"]) for l in log if "eval_loss" in l]
    ax1.plot(*zip(*ev), marker="o", markersize=3, label=f"{n_epochs}-epoch run")

    per_epoch = defaultdict(list)          # train loss is logged per step -> average per epoch
    for l in log:
        if "loss" in l:
            per_epoch[int(l["epoch"] - 1e-9) + 1].append(l["loss"])
    xs = sorted(per_epoch)
    ax2.plot(xs, [sum(per_epoch[x]) / len(per_epoch[x]) for x in xs], label=f"{n_epochs}-epoch run")

ax1.set(title="Eval loss (test split)", xlabel="epoch", ylabel="loss"); ax1.legend()
ax2.set(title="Train loss (epoch average)", xlabel="epoch", ylabel="loss"); ax2.legend()
plt.tight_layout()
plt.savefig("epoch_sweep.png", dpi=150)
plt.show()


In [ ]:
BEST_EPOCHS = 10   # <- set this after reading the sweep plot

import importlib, sweep_train
importlib.reload(sweep_train)
from sweep_train import build_model, make_trainer

for name in ("trainer", "model", "tokenizer"):
    globals().pop(name, None)
gc.collect()
torch.cuda.empty_cache()

model, tokenizer = build_model()
trainer = make_trainer(model, tokenizer, train_conv, eval_conv, BEST_EPOCHS, "outputs/final")


In [ ]:
import gc, torch
from transformers import TrainerCallback

class MemCB(TrainerCallback):
    def on_step_end(self, args, state, control, **kw):
        alloc = torch.cuda.memory_allocated() / 1e9
        peak  = torch.cuda.max_memory_allocated() / 1e9
        gc.collect(); torch.cuda.empty_cache()
        free  = torch.cuda.mem_get_info()[0] / 1e9
        print(f"step {state.global_step}/{state.max_steps} | alloc {alloc:.2f} GB | peak {peak:.2f} GB | free {free:.2f} GB")

trainer.add_callback(MemCB())

In [ ]:
import torch

free, total = torch.cuda.mem_get_info()

print(f"Free VRAM : {free/1024**3:.2f} GB")
print(f"Total VRAM: {total/1024**3:.2f} GB")

In [ ]:
print(trainer.args.per_device_train_batch_size)
print(trainer.args.gradient_accumulation_steps)
print(trainer.args.max_length)

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print(f"free {torch.cuda.mem_get_info()[0]/1e9:.2f} GB | reserved {torch.cuda.memory_reserved()/1e9:.2f} GB")

In [ ]:
import gc, torch

gc.collect()
torch.cuda.empty_cache()

trainer.args.max_steps = -1   # make sure the 3-step test limit is gone

trainer_stats = trainer.train()

In [ ]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()

batch = trainer.data_collator([converted_dataset[0]])
for k, v in batch.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))
print("prompt chars:", len(instruction))
print(f"free before: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

In [ ]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
FastVisionModel.for_inference(model) # Enable for inference!

image = Image.open(
    r"C:\Users\joel.sutanto\OneDrive - SK Jewellery\Pictures\jb.jpeg"
)

messages = [
    {"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": instruction}
    ]}
]
input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True)
inputs = tokenizer(
    image,
    input_text,
    add_special_tokens = False,
    return_tensors = "pt",
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128,
                   use_cache = True, temperature = 0, min_p = 0.7)

In [ ]:
print(len(test_dataset))

In [ ]:
import json
from tqdm.auto import tqdm

FastVisionModel.for_inference(model)  # must be called before generating

results = []

for i, sample in enumerate(tqdm(test_dataset, desc="Processing images")):
    messages = [{
        "role": "user",
        "content": [{"type": "image"}, {"type": "text", "text": instruction}],
    }]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

    inputs = tokenizer(
        sample["image"],
        input_text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        use_cache=True,
        do_sample=False,
    )

    gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    prediction = tokenizer.decode(gen_tokens, skip_special_tokens=True)

    # Check whether the output is valid JSON
    try:
        parsed = json.loads(prediction)
        valid_json = True
    except json.JSONDecodeError:
        parsed, valid_json = None, False

    results.append({
        "id": i,
        "prediction": prediction,
        "parsed": parsed,
        "valid_json": valid_json,
        "ground_truth": sample.get("label"),  # adjust to your field name
    })

with open("test_results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Valid JSON rate:", sum(r["valid_json"] for r in results) / len(results))

In [ ]:
print(len(results), len(test_dataset))   # should both be 76
print(results[0]["prediction"])
print(results[0]["ground_truth"])

In [ ]:
for r, sample in zip(results, test_dataset):
    r["ground_truth"] = json.loads(sample["text"])

print(results[0]["ground_truth"])

In [ ]:
import re
import pandas as pd

def loose(v):
    # lowercase, drop punctuation and spaces
    return re.sub(r"[^a-z0-9]", "", str(v or "").lower())

def get_fields(d):
    d = d if isinstance(d, dict) else {}
    return {
        "Id-no": loose(d.get("Id-no")),
        "Name": loose(d.get("Name")),
        "Address": loose("".join(str(d.get(k, "")) for k in ("Address_1", "Address_2", "Address_3"))),
        "Gender": loose(d.get("Gender")),
        "Date_Of_Birth": loose(d.get("Date_Of_Birth")),
    }

rows = []
for i, r in enumerate(results):
    gt, pr = get_fields(r["ground_truth"]), get_fields(r["parsed"])
    rows.append({"id": i, **{k: gt[k] == pr[k] for k in gt}})

df = pd.DataFrame(rows).set_index("id")

print((df.mean() * 100).round(1).to_string())
print(f"\nAll fields correct: {df.all(axis=1).mean():.1%}")

# Gender confusion matrix
y_true = [get_fields(r["ground_truth"])["Gender"] for r in results]
y_pred = [get_fields(r["parsed"])["Gender"] or "none" for r in results]
print(pd.crosstab(pd.Series(y_true, name="actual"), pd.Series(y_pred, name="predicted")))

In [ ]:
# pip install rapidfuzz scikit-learn matplotlib pandas
import re, json
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

def loose(v):
    return re.sub(r"[^a-z0-9]", "", str(v or "").lower())

def fields(d):
    d = d if isinstance(d, dict) else {}
    return {
        "Id-no": str(d.get("Id-no", "")),
        "Name": str(d.get("Name", "")),
        "Address": " ".join(str(d.get(k, "")) for k in ("Address_1", "Address_2", "Address_3")),
        "Gender": str(d.get("Gender", "")),
        "Date_Of_Birth": str(d.get("Date_Of_Birth", "")),
    }

rows = []
for i, r in enumerate(results):
    gt, pr = fields(r["ground_truth"]), fields(r["parsed"])
    for k in gt:
        g, p = gt[k], pr[k]
        rows.append({
            "id": i, "field": k,
            "exact": g.strip() == p.strip(),
            "loose": loose(g) == loose(p),
            "cer": Levenshtein.distance(loose(g), loose(p)) / max(len(loose(g)), 1),
        })

df = pd.DataFrame(rows)
summary = df.groupby("field").agg(
    exact=("exact", "mean"), loose=("loose", "mean"), cer=("cer", "mean")
)
summary[["exact", "loose"]] *= 100
summary["cer"] *= 100
print(summary.round(1))

# per-document-level: every field correct (loose)
per_img = df.pivot(index="id", columns="field", values="loose")
print(f"\nAll fields correct (loose): {per_img.all(axis=1).mean():.1%}")

# --- Plots ---
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))

summary[["exact", "loose"]].plot.bar(ax=ax[0], rot=30)
ax[0].set_title("Field accuracy (%)"); ax[0].set_ylim(0, 100)

summary["cer"].plot.bar(ax=ax[1], color="tab:red", rot=30)
ax[1].set_title("Character error rate (%) - lower is better")

y_true = [fields(r["ground_truth"])["Gender"].lower() for r in results]
y_pred = [(fields(r["parsed"])["Gender"].lower() or "none") for r in results]
labels = sorted(set(y_true) | set(y_pred))
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred, labels=labels),
                       display_labels=labels).plot(ax=ax[2], colorbar=False)
ax[2].set_title("Gender confusion matrix")

plt.tight_layout()
plt.savefig("eval_results.png", dpi=150)
plt.show()

In [ ]:
save_path = "joel_ocr_lora"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Saved LoRA adapter to {save_path}")

In [ ]:
model.save_pretrained_merged(
    "joel_ocr_merged",
    tokenizer,
    save_method="merged_16bit",
)

In [ ]:
import os

print(os.listdir("joel_ocr_merged")[:20])

In [ ]:
import shutil

shutil.make_archive(
    "joel_ocr_merged",
    "zip",
    "joel_ocr_merged"
)

print("ZIP created!")

In [ ]:
from pathlib import Path

print(Path("joel_ocr_merged.zip").resolve())

In [ ]:
from transformers import Qwen3_5ForConditionalGeneration, AutoProcessor

model = Qwen3_5ForConditionalGeneration.from_pretrained(
    r"C:\Users\joel.sutanto\100426\Data_ingestion_for-_OCR\joel_ocr_merged",
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(
    r"C:\Users\joel.sutanto\100426\Data_ingestion_for-_OCR\joel_ocr_merged"
)

print("Model loaded successfully!")

In [ ]:
print(model.config)
